# ai memory architecture

an ai that improves over time needs a memory layer, not just a bigger context window.

the loop: capture useful events, consolidate them into clean memories, recall the right memories for the current turn, and invalidate stale ones when facts change.

## core pieces

- **profile summary:** compact stable context.
- **temporal facts:** atomic subject-predicate-object records with `valid_at`, `invalid_at`, source, and optional embedding.
- **open loops:** unresolved threads that should resurface later.
- **style fingerprint:** lightweight interaction tendencies.
- **ambient recall:** small memory packet loaded before generation.
- **deep recall:** bounded `search_memory(query)` for specific past details.
- **consolidation:** background job that writes new facts and invalidates old ones.
- **episode store:** raw source history for audit and repair.

## flow

```mermaid
flowchart LR
    event[new event] --> history[recent history]
    event --> recall[ambient recall]
    recall --> mgraph[neo4j graph memory]
    recall --> vector[vector index]
    history --> generate[generation step]
    recall --> generate
    generate --> search{need exact past detail?}
    search -->|yes| tool[search memory]
    tool --> mgraph
    tool --> episodes[episode store]
    search -->|no| response[response]
    tool --> response
    response --> append[append episode]
    append --> consolidate[consolidation job]
    consolidate --> profile[profile summary]
    consolidate --> facts[temporal facts]
    consolidate --> invalidations[invalidations]
    profile --> mgraph
    facts --> mgraph
    invalidations --> mgraph
```

## memory layers

| layer | job | storage |
| --- | --- | --- |
| working context | current turn and recent events | app db or cache |
| profile | compact stable summary | neo4j `Profile` node |
| temporal fact | durable atomic claim | neo4j `MemoryFact` node |
| open loop | unresolved thread | `MemoryFact` with `kind='open_loop'` |
| episode | source of truth | postgres, object storage, or neo4j |
| style fingerprint | interaction tendencies | profile property or node |

the fact layer stays small. the episode layer keeps receipts.

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Literal

try:
    from neo4j import GraphDatabase
except ImportError:
    GraphDatabase = None


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
EMBEDDING_DIMENSIONS = int(os.getenv("EMBEDDING_DIMENSIONS", "1536"))

## neo4j graph shape

labels:

- `User`
- `Profile`
- `MemoryFact`
- `Entity`
- `Session`
- `Message`

relationships:

- `(User)-[:HAS_PROFILE]->(Profile)`
- `(User)-[:HAS_MEMORY]->(MemoryFact)`
- `(MemoryFact)-[:ABOUT]->(Entity)`
- `(MemoryFact)-[:SOURCE_SESSION]->(Session)`
- `(MemoryFact)-[:SUPERSEDED_BY]->(MemoryFact)`
- `(Session)-[:HAS_MESSAGE]->(Message)`

In [ ]:
NEO4J_SCHEMA_CYPHER = f"""
CREATE CONSTRAINT user_id IF NOT EXISTS
FOR (u:User) REQUIRE u.id IS UNIQUE;

CREATE CONSTRAINT profile_id IF NOT EXISTS
FOR (p:Profile) REQUIRE p.id IS UNIQUE;

CREATE CONSTRAINT memory_fact_id IF NOT EXISTS
FOR (m:MemoryFact) REQUIRE m.id IS UNIQUE;

CREATE CONSTRAINT entity_key IF NOT EXISTS
FOR (e:Entity) REQUIRE e.key IS UNIQUE;

CREATE CONSTRAINT session_id IF NOT EXISTS
FOR (s:Session) REQUIRE s.id IS UNIQUE;

CREATE INDEX memory_validity IF NOT EXISTS
FOR (m:MemoryFact) ON (m.user_id, m.invalid_at, m.valid_at);

CREATE FULLTEXT INDEX memory_fact_text IF NOT EXISTS
FOR (m:MemoryFact) ON EACH [m.fact, m.subject, m.predicate, m.object];

CREATE VECTOR INDEX memory_fact_embedding IF NOT EXISTS
FOR (m:MemoryFact) ON (m.embedding)
OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBEDDING_DIMENSIONS}, `vector.similarity_function`: 'cosine'}}}};
"""

print(NEO4J_SCHEMA_CYPHER)

In [ ]:
@dataclass
class MemoryFact:
    id: str
    user_id: str
    subject: str
    predicate: str
    object: str
    fact: str
    kind: Literal["fact", "preference", "open_loop", "style", "commitment"] = "fact"
    confidence: float = 0.7
    valid_at: str = field(default_factory=now_iso)
    invalid_at: str | None = None
    source_session_id: str | None = None
    embedding: list[float] | None = None
    attrs: dict[str, Any] = field(default_factory=dict)


@dataclass
class ConsolidationPatch:
    user_summary: str | None = None
    style_fingerprint: str | None = None
    new_facts: list[MemoryFact] = field(default_factory=list)
    invalidations: list[dict[str, Any]] = field(default_factory=list)

## consolidation

run consolidation after a session or batch closes.

outputs:

- profile summary update
- style fingerprint update
- new atomic facts
- invalidations for contradicted, resolved, or replaced facts

In [ ]:
CONSOLIDATION_INSTRUCTIONS = """
you are updating long-term memory for a learning ai system.

input:
- a closed session transcript.
- existing profile summary.
- existing style fingerprint.
- candidate existing facts that may be superseded.

return structured json with:
- profile_summary_update: null or compact bullets.
- style_fingerprint_update: null or compact behavioral description.
- new_facts: atomic subject/predicate/object/fact records.
- invalidations: fact ids with reason and optional replacement fact id.

rules:
- store only explicit, durable, useful information.
- prefer small facts over broad summaries.
- include temporal validity and source session.
- mark unresolved threads as kind='open_loop'.
- invalidate stale facts only when the new session clearly contradicts or resolves them.
- keep sensitive data behind explicit consent and a clear use case.
""".strip()

print(CONSOLIDATION_INSTRUCTIONS)

In [ ]:
UPSERT_MEMORY_FACT_CYPHER = """
MERGE (u:User {id: $user_id})
MERGE (s:Session {id: $source_session_id})
MERGE (m:MemoryFact {id: $id})
SET m.user_id = $user_id,
    m.subject = $subject,
    m.predicate = $predicate,
    m.object = $object,
    m.fact = $fact,
    m.kind = $kind,
    m.confidence = $confidence,
    m.valid_at = datetime($valid_at),
    m.invalid_at = CASE WHEN $invalid_at IS NULL THEN NULL ELSE datetime($invalid_at) END,
    m.attrs = $attrs
WITH u, s, m
MERGE (u)-[:HAS_MEMORY]->(m)
MERGE (m)-[:SOURCE_SESSION]->(s)
WITH m
UNWIND $entity_keys AS entity_key
MERGE (e:Entity {key: entity_key})
MERGE (m)-[:ABOUT]->(e)
RETURN m.id AS id
"""


def fact_to_params(fact: MemoryFact) -> dict[str, Any]:
    entity_keys = sorted({
        fact.subject.strip().lower(),
        fact.object.strip().lower(),
    } - {"", "user", "system"})
    return {
        "id": fact.id,
        "user_id": fact.user_id,
        "subject": fact.subject,
        "predicate": fact.predicate,
        "object": fact.object,
        "fact": fact.fact,
        "kind": fact.kind,
        "confidence": fact.confidence,
        "valid_at": fact.valid_at,
        "invalid_at": fact.invalid_at,
        "source_session_id": fact.source_session_id,
        "attrs": fact.attrs,
        "entity_keys": entity_keys,
    }


print(UPSERT_MEMORY_FACT_CYPHER)

## recall

default retrieval mix:

1. embed the current event.
2. pull top semantic matches from active facts.
3. pull keyword matches for exact names, projects, and dates.
4. expand one hop through graph entities.
5. add active open loops.
6. rerank by relevance, confidence, recency, and task fit.
7. format a small memory packet for generation.

In [ ]:
VECTOR_RECALL_CYPHER = """
// for newer neo4j versions, prefer the cypher search clause when available.
// the procedure form is shown because it is compact and widely documented.
CALL db.index.vector.queryNodes('memory_fact_embedding', $k, $query_embedding)
YIELD node AS m, score
WHERE m.user_id = $user_id AND m.invalid_at IS NULL
RETURN m.id AS id, m.fact AS fact, m.kind AS kind, m.valid_at AS valid_at, score
ORDER BY score DESC
LIMIT $k
"""

GRAPH_EXPANSION_CYPHER = """
MATCH (:MemoryFact {id: $memory_id})-[:ABOUT]->(e:Entity)<-[:ABOUT]-(neighbor:MemoryFact)
WHERE neighbor.user_id = $user_id AND neighbor.invalid_at IS NULL
RETURN DISTINCT neighbor.id AS id,
       neighbor.fact AS fact,
       neighbor.kind AS kind,
       neighbor.valid_at AS valid_at
ORDER BY neighbor.valid_at DESC
LIMIT $k
"""

OPEN_LOOP_CYPHER = """
MATCH (:User {id: $user_id})-[:HAS_MEMORY]->(m:MemoryFact)
WHERE m.invalid_at IS NULL AND m.kind = 'open_loop'
RETURN m.id AS id, m.fact AS fact, m.valid_at AS valid_at
ORDER BY m.valid_at DESC
LIMIT $k
"""

print(VECTOR_RECALL_CYPHER)
print(GRAPH_EXPANSION_CYPHER)
print(OPEN_LOOP_CYPHER)

In [ ]:
def format_memory_block(profile_summary: str, style_fingerprint: str, recalled: list[dict[str, Any]]) -> str:
    """format a small memory packet for the model."""
    lines = ["memory context. use only if directly relevant."]
    if profile_summary:
        lines.append(f"\nstable context:\n{profile_summary.strip()}")
    if style_fingerprint:
        lines.append(f"\nstyle signal:\n{style_fingerprint.strip()}")
    if recalled:
        lines.append("\nrecalled facts:")
        for item in recalled[:12]:
            prefix = "open loop" if item.get("kind") == "open_loop" else "fact"
            lines.append(f"- {prefix}: {item['fact']} (since {str(item.get('valid_at', 'unknown'))[:10]})")
    return "\n".join(lines)

## invalidation

never overwrite stale memory silently. mark the old fact invalid, optionally link it to the replacement, and keep the source trail.

examples:

- location changed: invalidate old `lives_in`, add new `lives_in`.
- thread resolved: invalidate the `open_loop`, optionally add the outcome.
- preference changed: invalidate old preference, add new preference.

In [ ]:
INVALIDATE_FACT_CYPHER = """
MATCH (old:MemoryFact {id: $old_fact_id, user_id: $user_id})
WHERE old.invalid_at IS NULL
SET old.invalid_at = datetime($invalid_at),
    old.invalidation_reason = $reason
WITH old
OPTIONAL MATCH (new:MemoryFact {id: $replacement_fact_id, user_id: $user_id})
FOREACH (_ IN CASE WHEN new IS NULL THEN [] ELSE [1] END |
    MERGE (old)-[:SUPERSEDED_BY]->(new)
)
RETURN old.id AS invalidated_id
"""

print(INVALIDATE_FACT_CYPHER)